# QMill Circuit Compression SDK Walkthrough (External)

Use this notebook to run a compression job with the QAS SDK.

Before you start:
- Set `QAS_BASE_URL` to `https://qas.qmill.com` (default in this notebook).
- Run `qas auth login --base-url https://qas.qmill.com` in a terminal.
- Optional: `QAS_NUM_GPUS`, `QAS_ITERATION_MINUTES`, `QAS_GATE_SET`, `QAS_HPC_MODE`.
- Restart the kernel after changing environment variables.


## Install the SDK in this kernel
Install the latest public SDK release from PyPI:

```bash
pip install -U qas-sdk
```


In [ ]:
import sys

print("Kernel Python:", sys.executable)
!{sys.executable} -m pip install -U qas-sdk

Check the environment variables


In [ ]:
import os
from textwrap import dedent

BASE_URL = os.getenv("QAS_BASE_URL", "https://qas-dev.qmill.com").rstrip("/")

print(
    dedent(
        f"""
        Using QAS environment: {BASE_URL}
        Auth source: local CLI session (`qas auth login`)
        """
    ).strip(),
)

## Compression Parameters
- `hpc_mode`: one of the available modes returned by `get_hpc_mode()`.
- `num_gpus`: number of GPUs for real backends; default is `1` when omitted.
- `iteration_time_minutes`: max runtime per iteration in minutes.
- `gate_set`: choose from the mode's `gate_set_options`.

Tip: prefer explicit parameters in code for users; env vars are handy for local testing.


Submit a compression job


In [ ]:
from qas_sdk import CompressionJobOptions, QASClient

client = QASClient(base_url=BASE_URL)

mod5_4_circuit = (
    """
    OPENQASM 2.0;
    include \"qelib1.inc\";
    qreg q[5];
    x q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[3],q[4];
    rz(-pi/4) q[4];
    cx q[0],q[4];
    rz(pi/4) q[4];
    cx q[3],q[4];
    rz(-pi/4) q[4];
    cx q[0],q[4];
    cx q[0],q[3];
    rz(-pi/4) q[3];
    cx q[0],q[3];
    rz(pi/4) q[0];
    rz(pi/4) q[3];
    rz(pi/4) q[4];
    cx q[3],q[4];
    rz(-pi/4) q[4];
    cx q[2],q[4];
    rz(pi/4) q[4];
    cx q[3],q[4];
    rz(-pi/4) q[4];
    cx q[2],q[4];
    cx q[2],q[3];
    rz(-pi/4) q[3];
    cx q[2],q[3];
    rz(pi/4) q[2];
    rz(pi/4) q[3];
    rz(pi/4) q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[3],q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[2],q[4];
    rz(-pi/4) q[4];
    cx q[1],q[4];
    rz(pi/4) q[4];
    cx q[2],q[4];
    rz(-pi/4) q[4];
    cx q[1],q[4];
    cx q[1],q[2];
    rz(-pi/4) q[2];
    cx q[1],q[2];
    rz(pi/4) q[1];
    rz(pi/4) q[2];
    rz(pi/4) q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[2],q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[1],q[4];
    rz(-pi/4) q[4];
    cx q[0],q[4];
    rz(pi/4) q[4];
    cx q[1],q[4];
    rz(-pi/4) q[4];
    cx q[0],q[4];
    cx q[0],q[1];
    rz(-pi/4) q[1];
    cx q[0],q[1];
    rz(pi/4) q[0];
    rz(pi/4) q[1];
    rz(pi/4) q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[1],q[4];
    cx q[0],q[4];
    """
).strip()

num_gpus = os.getenv("QAS_NUM_GPUS")
NUM_GPUS = int(num_gpus) if num_gpus and num_gpus.strip() else None
ITERATION_MINUTES = int(os.getenv("QAS_ITERATION_MINUTES", "45"))
GATE_SET = os.getenv("QAS_GATE_SET", "CX_RX_RZ")
HPC_MODE = os.getenv("QAS_HPC_MODE", "lumi_v2_0")

compression_kwargs = {}
if NUM_GPUS is not None:
    compression_kwargs["num_gpus"] = NUM_GPUS

options = CompressionJobOptions(
    iteration_time_minutes=ITERATION_MINUTES,
    gate_set=GATE_SET,
    hpc_mode=HPC_MODE,
)
if options.to_payload():
    compression_kwargs["options"] = options

if compression_kwargs:
    print(f"Using compression overrides: {compression_kwargs}")
job = client.submit_compression(mod5_4_circuit, **compression_kwargs)
job_id = job["job_id"]
print(f"Submitted job: {job_id}")

Optional status check


In [ ]:
import time

while True:
    job = client.get_compression_job(job_id)
    status = job.get("status")
    print("Status:", status)
    if job.get("logs"):
        print(job["logs"])
    if status in {"COMPLETED", "FAILED", "STOPPED", "CANCELLED"}:
        break
    time.sleep(10)

Optional stopping of compression job step


In [ ]:
# Fetch the latest partial results before stopping, then re-fetch after stopping.
_pre_stop = client.get_compression_job(job_id)
stopped = client.stop_compression_job(job_id)
_post_stop = client.get_compression_job(job_id)
print("Stop requested. Status:", stopped.get("status"))

Fetch the results


In [ ]:
final_job = client.get_compression_job(job_id)
print("Status:", final_job.get("status"))
print("Compressed circuit:", final_job.get("result"))
print("Transpiled circuit (optional):", final_job.get("transpiled_circuit"))
print("Compressed metrics (optional):", final_job.get("compressed_metrics"))
print("Transpiled metrics (optional):", final_job.get("transpiled_metrics"))